# 通过 CQL 结合向量嵌入、OpenAI 与 Cassandra / Astra DB 探索哲学

### CassIO 版本

在本快速入门中，你将学习如何使用 OpenAI 的向量嵌入，以及 [Apache Cassandra®](https://cassandra.apache.org)（或等效的 DataStax [Astra DB through CQL](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html)）作为数据持久化的向量存储，构建一个"哲学名言查找器与生成器"。

本 notebook 的基本工作流程概述如下。你将为一些著名哲学家的名言计算并存储向量嵌入，用它们构建一个强大的搜索引擎，之后甚至还能构建一个新名言的生成器！

本 notebook 展示了向量搜索的一些标准使用模式，同时也展示了上手 [Cassandra](https://cassandra.apache.org/doc/trunk/cassandra/vector-search/overview.html) / [Astra DB through CQL](https://docs.datastax.com/en/astra-serverless/docs/vector-search/quickstart.html) 的向量能力有多么简单。

如果想了解使用向量搜索和文本嵌入构建问答系统的背景知识，请查看这个出色的实践 notebook：[Question answering using embeddings](https://github.com/openai/openai-cookbook/blob/main/examples/Question_answering_using_embeddings.ipynb)。

#### _选择你的框架_

请注意，本 notebook 使用的是 [CassIO 库](https://cassio.org)，但我们也介绍了完成同一任务的其他技术选择。查看此文件夹的 [README](https://github.com/openai/openai-cookbook/tree/main/examples/vector_databases/cassandra_astradb) 了解其他选项。本 notebook 既可以作为 Colab notebook 运行，也可以作为普通 Jupyter notebook 运行。

目录：
- 设置
- 获取数据库连接
- 连接 OpenAI
- 将名言加载到向量存储中
- 用例 1：**名言搜索引擎**
- 用例 2：**名言生成器**
- （可选）在向量存储中利用分区

### 工作原理

**索引**

每条名言都通过 OpenAI 的 `Embedding` 转换为嵌入向量。这些向量会保存在向量存储中，供之后搜索使用。一些元数据（包括作者姓名和其他几个预先计算好的标签）也会一并存储，以便自定义搜索。

![1_向量索引](https://user-images.githubusercontent.com/14221764/282440878-dc3ed680-7d0e-4b30-9a74-d2d66a7394f7.png)

**搜索**

为了找到与给定搜索名言相似的名言，会即时将后者转换为嵌入向量，并用该向量在存储中查询相似的向量……也就是之前已建立索引的相似名言。搜索还可以选择性地通过额外的元数据加以约束（"帮我找出斯宾诺莎说过的、与这句相似的名言……"）。

![2_向量搜索](https://user-images.githubusercontent.com/14221764/282440908-683e3ee1-0bf1-46b3-8621-86c31fc7f9c9.png)

这里的关键在于，"内容相似的名言"在向量空间中对应于度量上彼此接近的向量：因此，向量相似度搜索实际上实现了语义相似度。_这正是向量嵌入如此强大的关键原因。_

下面的示意图试图传达这一思想。每条名言一旦被转换为向量，就是空间中的一个点。确切地说，在本例中它位于一个球面上，因为 OpenAI 的嵌入向量和大多数其他嵌入向量一样，都被归一化为_单位长度_。哦，而且这个球面其实不是三维的，而是 1536 维的！

所以，从本质上讲，向量空间中的相似度搜索返回的是与查询向量最接近的那些向量：

![3_向量空间](https://user-images.githubusercontent.com/14221764/262321363-c8c625c1-8be9-450e-8c68-b1ed518f990d.png)

**生成**

给定一个提示（一个主题或一句草拟的名言），先执行搜索步骤，然后将返回的前几条结果（名言）放入 LLM 提示中，让生成模型参照所传入的示例_以及_最初的提示创作一段新文本。

![4_名言生成](https://user-images.githubusercontent.com/14221764/282440927-d56f36eb-d611-4342-8026-7736edc6f5c9.png)

## 设置

首先安装一些必需的包：

In [1]:
!pip install --quiet "cassio>=0.1.3" "openai>=1.0.0" datasets

In [2]:
from getpass import getpass
from collections import Counter

import cassio
from cassio.table import MetadataVectorCassandraTable

import openai
from datasets import load_dataset

## 获取数据库连接

要通过 CQL 连接到你的 Astra DB，你需要两样东西：
- 一个具有 "Database Administrator" 角色的令牌（形如 `AstraCS:...`）
- 数据库 ID（形如 `3df2a5b6-...`）

    请确保你拥有这两个字符串——登录 [Astra UI](https://astra.datastax.com) 后即可获取。更多信息请参阅：[数据库 ID](https://awesome-astra.github.io/docs/pages/astra/faq/#where-should-i-find-a-database-identifier) 和 [令牌](https://awesome-astra.github.io/docs/pages/astra/create-token/#c-procedure)。

如果你想_连接到 Cassandra 集群_（不过该集群必须[支持](https://cassandra.apache.org/doc/trunk/cassandra/vector-search/overview.html)向量搜索），请将其替换为 `cassio.init(session=..., keyspace=...)`，并为你的集群使用合适的 session 和 keyspace 名称。

In [3]:
astra_token = getpass("Please enter your Astra token ('AstraCS:...')")
database_id = input("Please enter your database id ('3df2a5b6-...')")

Please enter your Astra token ('AstraCS:...') ········
Please enter your database id ('3df2a5b6-...') 01234567-89ab-dcef-0123-456789abcdef


In [4]:
cassio.init(token=astra_token, database_id=database_id)

### 创建数据库连接

以下是通过 CQL 创建 Astra DB 连接的方法：

_（顺便一提，你也可以使用任何 Cassandra 集群（只要它提供向量能力），只需[修改参数](https://docs.datastax.com/en/developer/python-driver/latest/getting_started/#connecting-to-cassandra)为下面的 `Cluster` 实例化所需的参数即可。）_

### 通过 CassIO 创建向量存储

你需要一个支持向量并带有元数据的表。将其命名为 "philosophers_cassio"：

In [5]:
v_table = MetadataVectorCassandraTable(table="philosophers_cassio", vector_dimension=1536)

## 连接 OpenAI

### 设置你的密钥

In [6]:
OPENAI_API_KEY = getpass("Please enter your OpenAI API Key: ")

Please enter your OpenAI API Key:  ········


### 嵌入的测试调用

快速看一下如何为一组输入文本获取嵌入向量：

In [7]:
client = openai.OpenAI(api_key=OPENAI_API_KEY)
embedding_model_name = "text-embedding-3-small"

result = client.embeddings.create(
    input=[
        "This is a sentence",
        "A second sentence"
    ],
    model=embedding_model_name,
)

_注意：以上是 OpenAI v1.0+ 的语法。如果使用之前的版本，获取嵌入的代码会有所不同。_

In [8]:
print(f"len(result.data)              = {len(result.data)}")
print(f"result.data[1].embedding      = {str(result.data[1].embedding)[:55]}...")
print(f"len(result.data[1].embedding) = {len(result.data[1].embedding)}")

len(result.data)              = 2
result.data[1].embedding      = [-0.010821706615388393, 0.001387271680869162, 0.0035479...
len(result.data[1].embedding) = 1536


## 将名言加载到向量存储中

_注意：以上是 OpenAI v1.0+ 的语法。如果使用之前的版本，获取嵌入的代码会有所不同。_

In [9]:
philo_dataset = load_dataset("datastax/philosopher-quotes")["train"]

快速查看一下：

In [10]:
print("An example entry:")
print(philo_dataset[16])

An example entry:
{'author': 'aristotle', 'quote': 'Love well, be loved and do something of value.', 'tags': 'love;ethics'}


检查数据集大小：

In [11]:
author_count = Counter(entry["author"] for entry in philo_dataset)
print(f"Total: {len(philo_dataset)} quotes. By author:")
for author, count in author_count.most_common():
    print(f"    {author:<20}: {count} quotes")

Total: 450 quotes. By author:
    aristotle           : 50 quotes
    schopenhauer        : 50 quotes
    spinoza             : 50 quotes
    hegel               : 50 quotes
    freud               : 50 quotes
    nietzsche           : 50 quotes
    sartre              : 50 quotes
    plato               : 50 quotes
    kant                : 50 quotes


### 将名言插入向量存储

你将为名言计算嵌入，并将它们连同文本本身以及计划之后使用的元数据一起保存到向量存储中。注意，作者会作为一个元数据字段，与名言本身已有的 "tags" 一起添加。

为了优化速度并减少调用次数，你将对 OpenAI 嵌入服务执行批量调用。

_（注意：为了更快地执行，Cassandra 和 CassIO 允许你进行并发插入，这里为了让演示代码更直观，我们没有这样做。）_

In [12]:
BATCH_SIZE = 50

num_batches = ((len(philo_dataset) + BATCH_SIZE - 1) // BATCH_SIZE)

quotes_list = philo_dataset["quote"]
authors_list = philo_dataset["author"]
tags_list = philo_dataset["tags"]

print("Starting to store entries:")
for batch_i in range(num_batches):
    b_start = batch_i * BATCH_SIZE
    b_end = (batch_i + 1) * BATCH_SIZE
    # compute the embedding vectors for this batch
    b_emb_results = client.embeddings.create(
        input=quotes_list[b_start : b_end],
        model=embedding_model_name,
    )
    # prepare the rows for insertion
    print("B ", end="")
    for entry_idx, emb_result in zip(range(b_start, b_end), b_emb_results.data):
        if tags_list[entry_idx]:
            tags = {
                tag
                for tag in tags_list[entry_idx].split(";")
            }
        else:
            tags = set()
        author = authors_list[entry_idx]
        quote = quotes_list[entry_idx]
        v_table.put(
            row_id=f"q_{author}_{entry_idx}",
            body_blob=quote,
            vector=emb_result.embedding,
            metadata={**{tag: True for tag in tags}, **{"author": author}},
        )
        print("*", end="")
    print(f" done ({len(b_emb_results.data)})")

print("\nFinished storing entries.")

Starting to store entries:
B ************************************************** done (50)
B ************************************************** done (50)
B ************************************************** done (50)
B ************************************************** done (50)
B ************************************************** done (50)
B ************************************************** done (50)
B ************************************************** done (50)
B ************************************************** done (50)
B ************************************************** done (50)

Finished storing entries.


## 用例 1：**名言搜索引擎**

对于名言搜索功能，你首先需要将输入的名言转换为向量，然后用它查询存储（此外还要在搜索调用中处理可选的元数据）。

将搜索引擎功能封装到一个函数中，以便复用：

In [13]:
def find_quote_and_author(query_quote, n, author=None, tags=None):
    query_vector = client.embeddings.create(
        input=[query_quote],
        model=embedding_model_name,
    ).data[0].embedding
    metadata = {}
    if author:
        metadata["author"] = author
    if tags:
        for tag in tags:
            metadata[tag] = True
    #
    results = v_table.ann_search(
        query_vector,
        n=n,
        metadata=metadata,
    )
    return [
        (result["body_blob"], result["metadata"]["author"])
        for result in results
    ]

### 测试搜索

只传入一条名言：

In [14]:
find_quote_and_author("We struggle all our life for nothing", 3)

[('Life to the great majority is only a constant struggle for mere existence, with the certainty of losing it at last.',
  'schopenhauer'),
 ('We give up leisure in order that we may have leisure, just as we go to war in order that we may have peace.',
  'aristotle'),
 ('Perhaps the gods are kind to us, by making life more disagreeable as we grow older. In the end death seems less intolerable than the manifold burdens we carry',
  'freud')]

限定作者的搜索：

In [15]:
find_quote_and_author("We struggle all our life for nothing", 2, author="nietzsche")

[('To live is to suffer, to survive is to find some meaning in the suffering.',
  'nietzsche'),
 ('What makes us heroic?--Confronting simultaneously our supreme suffering and our supreme hope.',
  'nietzsche')]

限定标签的搜索（从之前随名言一起保存的标签中选取）：

In [16]:
find_quote_and_author("We struggle all our life for nothing", 2, tags=["politics"])

[('Mankind will never see an end of trouble until lovers of wisdom come to hold political power, or the holders of power become lovers of wisdom',
  'plato'),
 ('Everything the State says is a lie, and everything it has it has stolen.',
  'nietzsche')]

### 剔除不相关的结果

向量相似度搜索通常会返回与查询最接近的向量，即使在没有更好结果的情况下，这意味着返回的结果可能有些不相关。

为了控制这个问题，你可以获取查询与每个结果之间的实际"距离"，然后为其设置一个阈值，从而有效地丢弃超出该阈值的结果。
正确调整这个阈值并不是一个简单的问题：这里我们只是向你展示做法。

为了感受其工作原理，请尝试下面的查询，并调整名言和阈值的选择来比较结果：

_注意（写给喜欢数学的读者）：这个"距离"正是向量之间的余弦相似度，即两个向量的标量积除以它们模长的乘积。因此，它是一个介于 -1 到 +1 之间的数，-1 表示方向完全相反的向量，+1 表示方向完全相同的向量。在其他地方（例如本演示的 "CQL" 版本中），你会得到该量经过重新缩放以适应 [0, 1] 区间的值，这意味着那里得到的数值和合适的阈值都会相应地变换。_

In [17]:
quote = "Animals are our equals."
# quote = "Be good."
# quote = "This teapot is strange."

metric_threshold = 0.84

quote_vector = client.embeddings.create(
    input=[quote],
    model=embedding_model_name,
).data[0].embedding

results = list(v_table.metric_ann_search(
    quote_vector,
    n=8,
    metric="cos",
    metric_threshold=metric_threshold,
))

print(f"{len(results)} quotes within the threshold:")
for idx, result in enumerate(results):
    print(f"    {idx}. [distance={result['distance']:.3f}] \"{result['body_blob'][:70]}...\"")

3 quotes within the threshold:
    0. [distance=0.855] "The assumption that animals are without rights, and the illusion that ..."
    1. [distance=0.843] "Animals are in possession of themselves; their soul is in possession o..."
    2. [distance=0.841] "At his best, man is the noblest of all animals; separated from law and..."


## 用例 2：**名言生成器**

对于这项任务，你需要 OpenAI 的另一个组件，即一个 LLM，来为我们生成名言（基于查询向量存储获得的输入）。

你还需要一个提示模板，用于为生成名言的 LLM 补全任务填充内容。

In [18]:
completion_model_name = "gpt-3.5-turbo"

generation_prompt_template = """"Generate a single short philosophical quote on the given topic,
similar in spirit and form to the provided actual example quotes.
Do not exceed 20-30 words in your quote.

REFERENCE TOPIC: "{topic}"

ACTUAL EXAMPLES:
{examples}
"""

和搜索一样，这个功能最好封装成一个方便的函数（其内部会使用搜索）：

In [19]:
def generate_quote(topic, n=2, author=None, tags=None):
    quotes = find_quote_and_author(query_quote=topic, n=n, author=author, tags=tags)
    if quotes:
        prompt = generation_prompt_template.format(
            topic=topic,
            examples="\n".join(f"  - {quote[0]}" for quote in quotes),
        )
        # a little logging:
        print("** quotes found:")
        for q, a in quotes:
            print(f"**    - {q} ({a})")
        print("** end of logging")
        #
        response = client.chat.completions.create(
            model=completion_model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=320,
        )
        return response.choices[0].message.content.replace('"', '').strip()
    else:
        print("** no quotes found.")
        return None

_注意：与嵌入计算的情况类似，对于 v1.0 之前的 OpenAI，Chat Completion API 的代码会略有不同。_

#### 测试名言生成

只传入一段文本（一条"名言"，但实际上也可以只提出一个主题，因为它的向量嵌入最终仍会落在向量空间中的正确位置）：

In [20]:
q_topic = generate_quote("politics and virtue")
print("\nA new generated quote:")
print(q_topic)

** quotes found:
**    - Happiness is the reward of virtue. (aristotle)
**    - Our moral virtues benefit mainly other people; intellectual virtues, on the other hand, benefit primarily ourselves; therefore the former make us universally popular, the latter unpopular. (schopenhauer)
** end of logging

A new generated quote:
Virtuous politics purifies society, while corrupt politics breeds chaos and decay.


只从一位哲学家那里汲取灵感：

In [21]:
q_topic = generate_quote("animals", author="schopenhauer")
print("\nA new generated quote:")
print(q_topic)

** quotes found:
**    - Because Christian morality leaves animals out of account, they are at once outlawed in philosophical morals; they are mere 'things,' mere means to any ends whatsoever. They can therefore be used for vivisection, hunting, coursing, bullfights, and horse racing, and can be whipped to death as they struggle along with heavy carts of stone. Shame on such a morality that is worthy of pariahs, and that fails to recognize the eternal essence that exists in every living thing, and shines forth with inscrutable significance from all eyes that see the sun! (schopenhauer)
**    - The assumption that animals are without rights, and the illusion that our treatment of them has no moral significance, is a positively outrageous example of Western crudity and barbarity. Universal compassion is the only guarantee of morality. (schopenhauer)
** end of logging

A new generated quote:
The true measure of humanity lies not in our dominion over animals, but in our ability to show com

## （可选）**分区**

在完成本快速入门之前，还有一个有趣的话题值得探讨。虽然一般来说，标签和名言之间可以是任意关系（例如一条名言有多个标签），但_作者_实际上是一种精确的分组（它们在名言集合上定义了一个"不相交划分"）：每条名言恰好只有一位作者（至少对我们而言是这样）。

现在，假设你事先知道你的应用通常（或总是）只针对_单个作者_运行查询。那么你就可以充分利用底层的数据库结构：如果你将名言按**分区**分组（每位作者一个分区），那么只针对某位作者的向量查询将使用更少的资源，并且返回得快得多。

我们不会在这里深入探讨其中的细节，因为它们涉及 Cassandra 存储的内部机制：重要的是，**如果你的查询是在某个分组内运行的，请考虑相应地进行分区以提升性能**。

现在你将看到这一选择的实际效果。

首先，你需要 CassIO 中另一种表抽象：

In [22]:
from cassio.table import ClusteredMetadataVectorCassandraTable

In [23]:
v_table_partitioned = ClusteredMetadataVectorCassandraTable(table="philosophers_cassio_partitioned", vector_dimension=1536)

现在在新表上重复计算嵌入并插入的步骤。

与之前相比，一个关键区别在于：现在名言的作者被存储为所插入行的_分区 id_，而不是被添加到包罗一切的 "metadata" 字典中。

趁此机会，作为演示，你将_并发地_插入某位作者的所有名言：在 CassIO 中，做法是对每条名言使用异步的 `put_async` 方法，收集得到的 `Future` 对象列表，之后对它们全部调用 `result()` 方法，以确保它们都已执行。Cassandra / Astra DB 很好地支持高并发的 I/O 操作。

_（注意：本可以缓存之前计算好的嵌入以节省一些 API 令牌——不过这里我们希望让代码更易于查看。）_

In [24]:
BATCH_SIZE = 50

num_batches = ((len(philo_dataset) + BATCH_SIZE - 1) // BATCH_SIZE)

quotes_list = philo_dataset["quote"]
authors_list = philo_dataset["author"]
tags_list = philo_dataset["tags"]

print("Starting to store entries:")
for batch_i in range(num_batches):
    b_start = batch_i * BATCH_SIZE
    b_end = (batch_i + 1) * BATCH_SIZE
    # compute the embedding vectors for this batch
    b_emb_results = client.embeddings.create(
        input=quotes_list[b_start : b_end],
        model=embedding_model_name,
    )
    # prepare the rows for insertion
    futures = []
    print("B ", end="")
    for entry_idx, emb_result in zip(range(b_start, b_end), b_emb_results.data):
        if tags_list[entry_idx]:
            tags = {
                tag
                for tag in tags_list[entry_idx].split(";")
            }
        else:
            tags = set()
        author = authors_list[entry_idx]
        quote = quotes_list[entry_idx]
        futures.append(v_table_partitioned.put_async(
            partition_id=author,
            row_id=f"q_{author}_{entry_idx}",
            body_blob=quote,
            vector=emb_result.embedding,
            metadata={tag: True for tag in tags},
        ))
    #
    for future in futures:
        future.result()
    #
    print(f" done ({len(b_emb_results.data)})")

print("\nFinished storing entries.")

Starting to store entries:
B  done (50)
B  done (50)
B  done (50)
B  done (50)
B  done (50)
B  done (50)
B  done (50)
B  done (50)
B  done (50)

Finished storing entries.


有了这个新表，相似度搜索也会相应变化（注意 `ann_search` 的参数）：

In [25]:
def find_quote_and_author_p(query_quote, n, author=None, tags=None):
    query_vector = client.embeddings.create(
        input=[query_quote],
        model=embedding_model_name,
    ).data[0].embedding
    metadata = {}
    partition_id = None
    if author:
        partition_id = author
    if tags:
        for tag in tags:
            metadata[tag] = True
    #
    results = v_table_partitioned.ann_search(
        query_vector,
        n=n,
        partition_id=partition_id,
        metadata=metadata,
    )
    return [
        (result["body_blob"], result["partition_id"])
        for result in results
    ]

就是这样：新表仍然完全支持"通用"的相似度搜索……

In [26]:
find_quote_and_author_p("We struggle all our life for nothing", 3)

[('Life to the great majority is only a constant struggle for mere existence, with the certainty of losing it at last.',
  'schopenhauer'),
 ('We give up leisure in order that we may have leisure, just as we go to war in order that we may have peace.',
  'aristotle'),
 ('Perhaps the gods are kind to us, by making life more disagreeable as we grow older. In the end death seems less intolerable than the manifold burdens we carry',
  'freud')]

……但只有在指定作者时，你才会注意到_巨大_的性能优势：

In [27]:
find_quote_and_author_p("We struggle all our life for nothing", 2, author="nietzsche")

[('To live is to suffer, to survive is to find some meaning in the suffering.',
  'nietzsche'),
 ('What makes us heroic?--Confronting simultaneously our supreme suffering and our supreme hope.',
  'nietzsche')]

嗯，如果你有一个规模接近真实场景的数据集，_就会_注意到性能提升。在这个演示中，只有几十条记录，看不出明显差异——但你应该明白其中的道理了。

## 总结

恭喜！你已经学会了如何使用 OpenAI 生成向量嵌入，并通过 CQL 使用 Cassandra / Astra DB 进行存储，从而构建一个精巧的哲学搜索引擎和名言生成器。

本示例使用 [CassIO](https://cassio.org) 与向量存储交互——但这并不是唯一的选择。查看 [README](https://github.com/openai/openai-cookbook/tree/main/examples/vector_databases/cassandra_astradb) 了解其他选项以及与流行框架的集成。

要进一步了解 Astra DB 的向量搜索能力如何成为你的 ML/GenAI 应用的关键要素，请访问 [Astra DB](https://docs.datastax.com/en/astra/home/astra.html) 关于该主题的网页。

## 清理

如果你想删除本演示使用的所有资源，请运行此单元（_警告：这将删除表以及插入其中的数据！_）：

In [28]:
# we peek at CassIO's config to get a direct handle to the DB connection
session = cassio.config.resolve_session()
keyspace = cassio.config.resolve_keyspace()

session.execute(f"DROP TABLE IF EXISTS {keyspace}.philosophers_cassio;")
session.execute(f"DROP TABLE IF EXISTS {keyspace}.philosophers_cassio_partitioned;")